<a href="https://colab.research.google.com/github/amribanerjee/VoxelSynth-3D/blob/main/patient1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# Install pydicom if not already installed
!pip install pydicom

import pydicom
import numpy as np
import os
from pathlib import Path
from scipy.ndimage import median_filter, zoom

class CTPreprocessor:
    def __init__(self, output_root_path):
        self.output_root_path = Path(output_root_path)
        self.volume = None
        self.spacing = None

    def validate_and_load(self, patient_dir):
        # Load all .dcm files in the patient directory
        files = [pydicom.dcmread(f) for f in Path(patient_dir).glob("*.dcm")]

        # Filter for Axial slices to ensure volumetric consistency
        files = [f for f in files if 'ImageType' in f and "AXIAL" in str(f.ImageType).upper()]

        if not files:
            print(f"No valid axial slices found in {patient_dir}")
            return None

        # Sort by Z-axis position (ImagePositionPatient[2]) to ensure the 266 slices are in order
        files.sort(key=lambda x: float(x.ImagePositionPatient[2]))

        self.spacing = [
            float(files[0].SliceThickness),
            float(files[0].PixelSpacing[0]),
            float(files[0].PixelSpacing[1])
        ]

        # Stack the 2D pixel arrays into a 3D volume
        self.volume = np.stack([f.pixel_array for f in files])

        # Convert to Hounsfield Units (HU) using rescale slope and intercept
        slope = files[0].RescaleSlope if 'RescaleSlope' in files[0] else 1
        intercept = files[0].RescaleIntercept if 'RescaleIntercept' in files[0] else 0
        self.volume = self.volume.astype(np.float32) * slope + intercept

        return self.volume

    def apply_transforms(self, target_spacing=[1.0, 1.0, 1.0]):
        if self.volume is None:
            return None

        # Apply 3D median filter for noise reduction (denoising before resampling)
        self.volume = median_filter(self.volume, size=3)

        # Calculate resampling factors for isotropic voxel size (1.0mm^3)
        factors = [c/t for c, t in zip(self.spacing, target_spacing)]
        self.volume = zoom(self.volume, factors, order=3)
        self.spacing = target_spacing
        return self.volume

    def save_volume(self, patient_folder_name):
        # Create the specific patient subfolder in data/processed/
        patient_output_dir = self.output_root_path / patient_folder_name
        patient_output_dir.mkdir(parents=True, exist_ok=True)

        save_path = patient_output_dir / f"{patient_folder_name}_phi.npy"
        np.save(save_path, self.volume)
        print(f"Successfully processed and saved: {save_path}")

if __name__ == "__main__":
    # Path configuration relative to script location
    base_dir = Path.cwd()
    raw_root = base_dir / 'data' / 'raw'
    proc_root = base_dir / 'data' / 'processed'

    engine = CTPreprocessor(output_root_path=proc_root)

    if raw_root.exists():
        for patient_folder in raw_root.iterdir(): # patient_folder here is e.g. data/raw/patient1
            if patient_folder.is_dir():
                # Check for a nested directory if the structure is data/raw/patient1/patient1
                actual_dicom_dir = patient_folder
                nested_subdirs = [d for d in patient_folder.iterdir() if d.is_dir()]
                # If there's exactly one subdirectory and its name matches the parent patient folder, use it
                if len(nested_subdirs) == 1 and nested_subdirs[0].name == patient_folder.name:
                    actual_dicom_dir = nested_subdirs[0]

                print(f"Processing {actual_dicom_dir.name}...")
                vol = engine.validate_and_load(actual_dicom_dir)
                if vol is not None:
                    engine.apply_transforms()
                    # Saves to data/processed/patient1/patient1_phi.npy
                    engine.save_volume(actual_dicom_dir.name)

Processing patient1...
Successfully processed and saved: /content/data/processed/patient1/patient1_phi.npy


In [7]:
import zipfile
import os
from pathlib import Path

# Define the path to the uploaded zip file
zip_file_path = Path('patient1.zip')

# Define the directory where the DICOM files should be extracted
# This assumes the raw data should be in data/raw/patient1
output_dir = Path('data') / 'raw' / 'patient1'

# Create the output directory if it doesn't exist
output_dir.mkdir(parents=True, exist_ok=True)

# Unzip the file
if zip_file_path.exists():
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(output_dir)
    print(f"Successfully unzipped {zip_file_path} to {output_dir}")
    # Optionally, remove the zip file after extraction
    # os.remove(zip_file_path)
else:
    print(f"Error: {zip_file_path} not found.")


Successfully unzipped patient1.zip to data/raw/patient1


In [9]:
# Verify the content of the extracted directory
print(f"Contents of {output_dir}:")
for item in output_dir.iterdir():
    print(item.name)


Contents of data/raw/patient1:
patient1


In [15]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from scipy.ndimage import sobel
from pathlib import Path

class ArtifactSegmenter:
    def __init__(self, max_depth=15):
        self.model = DecisionTreeClassifier(max_depth=max_depth)
        self.is_trained = False

    def _get_features(self, volume):
        dx = sobel(volume, axis=0)
        dy = sobel(volume, axis=1)
        dz = sobel(volume, axis=2)
        grad_mag = np.sqrt(dx**2 + dy**2 + dz**2).astype(np.float32)
        return np.stack([volume.flatten(), grad_mag.flatten()], axis=-1)

    def predict_mask(self, volume):
        if not self.is_trained:
            return (volume > 2500).astype(np.uint8)
        X = self._get_features(volume)
        prediction = self.model.predict(X)
        return prediction.reshape(volume.shape).astype(np.uint8)
if __name__ == "__main__":
    base_dir = Path.cwd()
    proc_dir = base_dir / 'data' / 'processed'

    segmenter = ArtifactSegmenter()

    for npy_file in proc_dir.glob("*_phi.npy"):
        vol = np.load(npy_file)
        mask = segmenter.predict_mask(vol)
        mask_path = npy_file.with_name(npy_file.stem.replace("_phi", "_mask") + ".npy")
        np.save(mask_path, mask)
        print(f"Generated mask for: {npy_file.name}")

In [16]:
if __name__ == "__main__":
    # Path setup
    base_dir = Path.cwd()
    proc_dir = base_dir / 'data' / 'processed'

    segmenter = ArtifactSegmenter()

    # Use rglob to find files inside data/processed/patient1/
    for npy_file in proc_dir.rglob("*_phi.npy"):
        vol = np.load(npy_file)

        # Generates the mask using your 3D gradient logic
        mask = segmenter.predict_mask(vol)

        # Correctly names the mask file within the same patient folder
        mask_name = npy_file.name.replace("_phi.npy", "_mask.npy")
        mask_path = npy_file.parent / mask_name

        np.save(mask_path, mask)
        print(f"Generated mask for: {npy_file.parent.name}/{npy_file.name}")

Generated mask for: patient1/patient1_phi.npy


In [20]:
import numpy as np
from pathlib import Path
from scipy.ndimage import distance_transform_edt

class VolumeSynthesizer:
    def __init__(self, blend_factor=0.7):
        self.blend_factor = blend_factor

    def linear_interpolation_fill(self, volume, mask):
        corrected_volume = volume.copy()
        for i in range(volume.shape[0]):
            slice_data = volume[i]
            slice_mask = mask[i]
            if np.any(slice_mask):
                valid_data = slice_data[slice_mask == 0]
                fill_val = np.median(valid_data) if valid_data.size > 0 else 0
                slice_data[slice_mask == 1] = fill_val
                corrected_volume[i] = slice_data
        return corrected_volume

    def synthesize_final(self, original_vol, processed_vol, mask):
        return np.where(mask == 1, processed_vol, original_vol)

if __name__ == "__main__":
    base_dir = Path.cwd()
    proc_dir = base_dir / 'data' / 'processed'

    synthesizer = VolumeSynthesizer()

    # Use rglob to find files inside data/processed/patient1/
    for phi_file in proc_dir.rglob("*_phi.npy"):
        mask_file = Path(str(phi_file).replace("_phi.npy", "_mask.npy"))

        if mask_file.exists():
            vol = np.load(phi_file)
            mask = np.load(mask_file)

            interp_vol = synthesizer.linear_interpolation_fill(vol, mask)
            final_output = synthesizer.synthesize_final(vol, interp_vol, mask)

            output_path = Path(str(phi_file).replace("_phi.npy", "_final.npy"))
            np.save(output_path, final_output)
            print(f"Final Synthesis complete for: {phi_file.name}")

Final Synthesis complete for: patient1_phi.npy
